<a href="https://colab.research.google.com/github/Fantiflex/Cognitive_biaises/blob/main/03_generate_nonsocial_matrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


**Imports**

In [2]:
from pathlib import Path
import re
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

In [3]:
base_dir = Path("/content/drive/MyDrive/Colab_notebooks/Ulm")

SOCIAL_DIR = base_dir / "matrices" / "social"
NONSOCIAL_DIR = base_dir / "matrices" / "nonsocial_matched"
NONSOCIAL_MANIFEST_CSV = base_dir / "matrices" / "nonsocial_matched_manifest.csv"

NONSOCIAL_DIR.mkdir(parents=True, exist_ok=True)

# **HELPER FUNCTIONS**

In [17]:
######## PARAMETERS ##########
CELL_PX=100
GRID_COLS=10
GRID_ROWS=10
CIRCLE_RADIUS_PCT=20
JPEG_QUAL = 82

In [18]:
def assemble_grid(cells):
    matrix = Image.new("RGB", (GRID_COLS * CELL_PX, GRID_ROWS * CELL_PX))

    for idx, cell in enumerate(cells):
        row = idx // GRID_COLS
        col = idx % GRID_COLS

        x = col * CELL_PX
        y = row * CELL_PX

        matrix.paste(cell, (x, y))

    return matrix

def make_luminance_matched_circle_cell(cell_px, gray_value):
    gray_value = int(np.clip(round(gray_value), 0, 255))

    img = Image.new("L", (cell_px, cell_px), color=255)
    draw = ImageDraw.Draw(img)

    r = round(CIRCLE_RADIUS_PCT * cell_px)
    cx = cell_px // 2
    cy = cell_px // 2

    bbox = [cx - r, cy - r, cx + r, cy + r]
    draw.ellipse(bbox, fill=gray_value, outline=gray_value)

    return img


def make_nonsocial_from_cell_info(cell_info, cell_px):
    cells = []

    cell_info = cell_info.sort_values("cell_idx")

    for _, row in cell_info.iterrows():
        gray = row["face_luminance"]
        cell = make_luminance_matched_circle_cell(cell_px, gray)
        cells.append(cell)

    matrix = assemble_grid(cells)

    return matrix

# **SAVE NONSOCIAL MATRICES IN SUBFOLDERS (10pct, 20pct, ...)**

In [20]:
# Dossier de sortie
NONSOCIAL_DIR = base_dir / "matrices" / "nonsocial"
NONSOCIAL_DIR.mkdir(parents=True, exist_ok=True)

# Créer les sous-dossiers selon les pourcentages présents
pct_values = sorted(social_cell_manifest["true_minority_pct"].astype(int).unique())

for pct in pct_values:
    (NONSOCIAL_DIR / f"{pct}pct").mkdir(parents=True, exist_ok=True)

# Optionnel : supprimer les anciennes images .jpg déjà présentes
for old_file in NONSOCIAL_DIR.rglob("*.jpg"):
    old_file.unlink()

# Liste pour garder une trace des fichiers sauvegardés
saved_rows = []

# Boucle sur chaque matrice sociale
for social_filename, group_df in social_cell_manifest.groupby("social_filename"):
    group_df = group_df.sort_values("cell_idx").copy()

    trial_idx = int(group_df["trial_idx"].iloc[0])
    pct = int(group_df["true_minority_pct"].iloc[0])
    category = group_df["category"].iloc[0]

    # Générer la matrice non sociale à partir des luminances cellule par cellule
    nonsocial_img = make_nonsocial_from_cell_info(
        cell_info=group_df,
        cell_px=CELL_PX
    )

    # Sous-dossier de sauvegarde (ex: 10pct, 20pct, ...)
    out_dir = NONSOCIAL_DIR / f"{pct}pct"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Nom du fichier
    out_name = f"T{trial_idx:03d}_nonsocial_{pct}pct.jpg"
    out_path = out_dir / out_name

    # Sauvegarde
    nonsocial_img.convert("RGB").save(
        out_path,
        format="JPEG",
        quality=90
    )

    # Garder les infos
    saved_rows.append({
        "source_social_filename": social_filename,
        "nonsocial_filename": out_name,
        "trial_idx": trial_idx,
        "category": category,
        "true_minority_pct": pct,
        "output_folder": str(out_dir),
        "output_path": str(out_path)
    })

# Créer un petit manifest de sortie
nonsocial_manifest = pd.DataFrame(saved_rows)

manifest_path = NONSOCIAL_DIR / "nonsocial_manifest.csv"
nonsocial_manifest.to_csv(manifest_path, index=False)

print(f"{len(nonsocial_manifest)} matrices non sociales sauvegardées.")
print(f"Manifest sauvegardé ici : {manifest_path}")

display(nonsocial_manifest.head())

40 matrices non sociales sauvegardées.
Manifest sauvegardé ici : /content/drive/MyDrive/Colab_notebooks/Ulm/matrices/nonsocial/nonsocial_manifest.csv


,source_social_filename,nonsocial_filename,trial_idx,category,true_minority_pct,output_folder,output_path
0,T001_men_30pct.jpg,T001_nonsocial_30pct.jpg,1,Men,30,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...
1,T002_white_10pct.jpg,T002_nonsocial_10pct.jpg,2,White,10,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...
2,T003_men_20pct.jpg,T003_nonsocial_20pct.jpg,3,Men,20,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...
3,T004_black_10pct.jpg,T004_nonsocial_10pct.jpg,4,Black,10,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...
4,T005_black_50pct.jpg,T005_nonsocial_50pct.jpg,5,Black,50,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...,/content/drive/MyDrive/Colab_notebooks/Ulm/mat...
